In [8]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import duckdb
con = duckdb.connect('artifacts/analysis.duckdb', read_only=True)

# Schema + row counts
print(con.execute("DESCRIBE SELECT * FROM 'artifacts/parquet/ads.parquet'").fetchdf())
print(con.execute("""
    SELECT profile, 
           COUNT(*) AS ads,
           COUNT(DISTINCT visit_id) AS visits_with_ads,
           COUNT(DISTINCT page_url) AS unique_pages,
           SUM(CASE WHEN ocr_text IS NOT NULL AND LENGTH(ocr_text) > 10 THEN 1 ELSE 0 END) AS ads_with_text
    FROM 'artifacts/parquet/ads.parquet'
    GROUP BY profile
""").fetchdf())
con.execute("""
    COPY (SELECT visit_id, ad_hash, ocr_text FROM 'artifacts/parquet/ads.parquet' WHERE ocr_text IS NOT NULL AND LENGTH(ocr_text) > 10) TO 'ads.csv' (HEADER, DELIMITER ',');
""")

            column_name column_type null   key default extra
0               profile     VARCHAR  YES  None    None  None
1              visit_id      BIGINT  YES  None    None  None
2              page_url     VARCHAR  YES  None    None  None
3               ad_hash     VARCHAR  YES  None    None  None
4                ad_src     VARCHAR  YES  None    None  None
5                ad_tag     VARCHAR  YES  None    None  None
6                 ad_id     VARCHAR  YES  None    None  None
7              ad_width      DOUBLE  YES  None    None  None
8             ad_height      DOUBLE  YES  None    None  None
9                  ad_x      DOUBLE  YES  None    None  None
10                 ad_y      DOUBLE  YES  None    None  None
11   advertiser_network     VARCHAR  YES  None    None  None
12      capture_network     VARCHAR  YES  None    None  None
13  verification_source     VARCHAR  YES  None    None  None
14       networks_agree     BOOLEAN  YES  None    None  None
15           confidence 